use comments and product features

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
data_path = os.getenv('DATA_PATH')
products = pd.read_csv(f'{data_path}/BaSalam.products.csv')
reviews = pd.read_csv(f'{data_path}/BaSalam.reviews.csv')
labeled_comments = pd.read_csv(f"{data_path}/all_labeled_comments.csv")
map_category = pd.read_csv(f"{data_path}/category_map.csv")
category_mapping = dict(zip(map_category['sub_category'], map_category['main_category']))
products['main_category'] = products['categoryTitle'].map(category_mapping).fillna("سایر")
products[['categoryTitle','main_category']].sample()

In [ ]:
users_id = reviews.drop_duplicates(subset=['_id', 'user_id'])['user_id'].value_counts().head(1000).index.values.tolist()
products_id = reviews.drop_duplicates(subset=['_id', 'user_id'])['productId'].value_counts().head(100000).index.values.tolist()

In [ ]:
data = reviews[(reviews['user_id'].isin(users_id)) & (reviews['productId'].isin(products_id)
                                                      )].drop_duplicates(subset=['productId', 'user_id']).reset_index()

In [ ]:
table = pd.pivot_table(data, values='star', index='user_id', columns='productId', fill_value=-1)

In [ ]:
drop_cols = table.loc[: ,table.sum() < -(table.shape[0]-3)].columns.values.tolist()
matrix = table.drop(drop_cols, axis=1)

In [ ]:
cosine_sim = cosine_similarity(matrix)

def predict_rating(user_index, item_index):
    similar_users = cosine_sim[user_index]
    weighted_sum = 0
    total_similarity = 0
    
    for i, sim in enumerate(similar_users):
        if matrix.iloc[i, item_index] != -1:  
            weighted_sum += sim * matrix.iloc[i, item_index]
            total_similarity += abs(sim)
    
    if total_similarity == 0:  
        return -1  
    return weighted_sum / total_similarity

def recommend(user_index):
    predictions = []
    for item_index in range(matrix.shape[1]): 
        if matrix.iloc[user_index, item_index] == -1:  
            predicted_rating = predict_rating(user_index, item_index)
            predictions.append((item_index, predicted_rating))
    
    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:20]  

In [ ]:
recommended_items = recommend(17)

print("Recommended items for user 17:")
for item, rating in recommended_items:
    print(f"Product {item} with predicted rating {rating:.2f}")

In [ ]:
matrix.iloc[:, 6352].name

In [ ]:
products[products['_id']==495145][['name', 'main_category', 'categoryTitle','rating_average']]

In [ ]:
id = matrix.index[17]

In [ ]:
pro_ids = reviews[reviews['user_id']== id]['productId']
products[products['_id'].isin(pro_ids)][['name', 'main_category', 'categoryTitle','rating_average']]

In [ ]:
products[products['_id'].isin(pro_ids)]['main_category'].value_counts()